In [2]:
from incidentiq.search import SearchEngine
from incidentiq.context.builder import ContextBuilder

e:\incidentiq\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DATA_PATH = "../data/processed/logs.parquet"

engine = SearchEngine(DATA_PATH)

context_builder = ContextBuilder(
    engine.df
)

Batches: 100%|██████████| 63/63 [00:04<00:00, 14.37it/s]


In [4]:
query = "node card failure"

results = engine.search_hybrid(
    query,
    top_k=10
)

In [5]:
evidence = context_builder.build(
    results
)

In [6]:
for item in evidence:

    print(
        f"DOC: {item['doc_id']}"
    )

    print(
        f"TIME: {item['timestamp']}"
    )

    print(
        f"NODE: {item['node']}"
    )

    print(
        f"SEVERITY: {item['severity']}"
    )

    print(
        f"MESSAGE: {item['message']}"
    )

    print("-" * 70)

DOC: 457
TIME: 2005-06-28 09:53:39.479164
NODE: R02-M1-NE
SEVERITY: WARNING
MESSAGE: Node card is not fully functional
----------------------------------------------------------------------
DOC: 1948
TIME: 2005-12-06 10:05:04.300635
NODE: R12-M0-NC
SEVERITY: WARNING
MESSAGE: Node card is not fully functional
----------------------------------------------------------------------
DOC: 1229
TIME: 2005-08-09 10:53:17.485009
NODE: R74-M0-N1
SEVERITY: WARNING
MESSAGE: Node card is not fully functional
----------------------------------------------------------------------
DOC: 1227
TIME: 2005-08-09 10:40:46.749252
NODE: R51-M1-ND
SEVERITY: WARNING
MESSAGE: Node card is not fully functional
----------------------------------------------------------------------
DOC: 1218
TIME: 2005-08-04 10:58:22.010246
NODE: R06-M1-ND
SEVERITY: WARNING
MESSAGE: Node card is not fully functional
----------------------------------------------------------------------
DOC: 620
TIME: 2005-07-08 23:15:03.798449
NODE

In [7]:
grouped_evidence = context_builder.group_by_message(
    evidence
)

In [8]:
for group in grouped_evidence:

    print(
        f"MESSAGE: {group['message']}"
    )

    print(
        f"OCCURRENCES: {group['count']}"
    )

    for occurrence in group["occurrences"]:

        print(
            f"  DOC={occurrence['doc_id']} | "
            f"TIME={occurrence['timestamp']} | "
            f"NODE={occurrence['node']} | "
            f"SEVERITY={occurrence['severity']}"
        )

    print("-" * 70)

MESSAGE: Node card is not fully functional
OCCURRENCES: 6
  DOC=457 | TIME=2005-06-28 09:53:39.479164 | NODE=R02-M1-NE | SEVERITY=WARNING
  DOC=1948 | TIME=2005-12-06 10:05:04.300635 | NODE=R12-M0-NC | SEVERITY=WARNING
  DOC=1229 | TIME=2005-08-09 10:53:17.485009 | NODE=R74-M0-N1 | SEVERITY=WARNING
  DOC=1227 | TIME=2005-08-09 10:40:46.749252 | NODE=R51-M1-ND | SEVERITY=WARNING
  DOC=1218 | TIME=2005-08-04 10:58:22.010246 | NODE=R06-M1-ND | SEVERITY=WARNING
  DOC=620 | TIME=2005-07-08 23:15:03.798449 | NODE=R05-M0-N2 | SEVERITY=WARNING
----------------------------------------------------------------------
MESSAGE: Node card VPD check: U01 node in processor card slot J05 do not match. VPD ecid 04D37DF2DE7BFFFF0D081AF0DAD2, found 04DD80740E2FFFFF0A0C19D0CEBD
OCCURRENCES: 1
  DOC=1801 | TIME=2005-11-29 16:21:56.855025 | NODE=R67-M1-N7 | SEVERITY=INFO
----------------------------------------------------------------------
MESSAGE: Can not get assembly information for node card
OCCURRENCES: 

In [9]:
timeline = context_builder.build_timeline(evidence)

In [10]:
for event in timeline:
    print(
        f"{event["timestamp"]} |"
        f"{event["node"]} |"
        f"{event["severity"]} |"
        f"{event["message"]}"
    )
    

2005-06-28 09:53:39.479164 |R02-M1-NE |WARNING |Node card is not fully functional
2005-07-01 11:05:31.120732 |R37-M1-N4 |SEVERE |Can not get assembly information for node card
2005-07-08 23:15:03.798449 |R05-M0-N2 |WARNING |Node card is not fully functional
2005-08-02 21:15:36.811548 |UNKNOWN_LOCATION |SEVERE |Can not get assembly information for node card
2005-08-04 10:58:22.010246 |R06-M1-ND |WARNING |Node card is not fully functional
2005-08-09 10:40:46.749252 |R51-M1-ND |WARNING |Node card is not fully functional
2005-08-09 10:53:17.485009 |R74-M0-N1 |WARNING |Node card is not fully functional
2005-09-20 11:57:30.636832 |R05-M0-NA |INFO |Node card VPD check: U01 node in processor card slot J15 do not match. VPD ecid 04DE7DB80D7BFFFF04051B70D8D9, found 04DE7DF2D37BFFFF09081B6088D9
2005-11-29 16:21:56.855025 |R67-M1-N7 |INFO |Node card VPD check: U01 node in processor card slot J05 do not match. VPD ecid 04D37DF2DE7BFFFF0D081AF0DAD2, found 04DD80740E2FFFFF0A0C19D0CEBD
2005-12-06 10:0

In [11]:
context = context_builder.build_context(
    results,
    max_evidence=10
)

In [12]:
print(
    "Evidence:",
    len(context["evidence"])
)

print(
    "Groups:",
    len(context["groups"])
)

print(
    "Timeline:",
    len(context["timeline"])
)

Evidence: 10
Groups: 4
Timeline: 10


In [13]:
print(context)
print(len(context["temporal_clusters"]))

{'evidence': [{'doc_id': 457, 'timestamp': Timestamp('2005-06-28 09:53:39.479164'), 'node': 'R02-M1-NE', 'severity': 'WARNING', 'message': 'Node card is not fully functional'}, {'doc_id': 1948, 'timestamp': Timestamp('2005-12-06 10:05:04.300635'), 'node': 'R12-M0-NC', 'severity': 'WARNING', 'message': 'Node card is not fully functional'}, {'doc_id': 1229, 'timestamp': Timestamp('2005-08-09 10:53:17.485009'), 'node': 'R74-M0-N1', 'severity': 'WARNING', 'message': 'Node card is not fully functional'}, {'doc_id': 1227, 'timestamp': Timestamp('2005-08-09 10:40:46.749252'), 'node': 'R51-M1-ND', 'severity': 'WARNING', 'message': 'Node card is not fully functional'}, {'doc_id': 1218, 'timestamp': Timestamp('2005-08-04 10:58:22.010246'), 'node': 'R06-M1-ND', 'severity': 'WARNING', 'message': 'Node card is not fully functional'}, {'doc_id': 620, 'timestamp': Timestamp('2005-07-08 23:15:03.798449'), 'node': 'R05-M0-N2', 'severity': 'WARNING', 'message': 'Node card is not fully functional'}, {'do

In [14]:
from incidentiq.context.signals import (
    extract_statistics
)

In [15]:
statistics = extract_statistics(
    context
)

statistics

{'total_events': 10,
 'unique_messages': 4,
 'affected_nodes': 10,
 'severity_distribution': {'WARNING': 6, 'INFO': 2, 'SEVERE': 2}}

In [16]:
from incidentiq.context.signals import(
    extract_time_range
)

In [17]:
time_range = extract_time_range(
    context
)

time_range

{'first_event': Timestamp('2005-06-28 09:53:39.479164'),
 'last_event': Timestamp('2005-12-06 10:05:04.300635'),
 'duration': {Timedelta('161 days 00:11:24.821471')}}

In [18]:
from incidentiq.context.signals import (
    cluster_events_by_time
)

In [19]:
clusters = cluster_events_by_time(
    context
)

In [20]:
for i, cluster in enumerate(
    clusters,
    start=1
):

    print(
        f"\nCLUSTER {i}"
    )

    print(
        f"EVENTS: {len(cluster)}"
    )

    for event in cluster:

        print(
            f"  {event['timestamp']} | "
            f"{event['node']} | "
            f"{event['severity']} | "
            f"{event['message']}"
        )


CLUSTER 1
EVENTS: 1
  2005-06-28 09:53:39.479164 | R02-M1-NE | WARNING | Node card is not fully functional

CLUSTER 2
EVENTS: 1
  2005-07-01 11:05:31.120732 | R37-M1-N4 | SEVERE | Can not get assembly information for node card

CLUSTER 3
EVENTS: 1
  2005-07-08 23:15:03.798449 | R05-M0-N2 | WARNING | Node card is not fully functional

CLUSTER 4
EVENTS: 1
  2005-08-02 21:15:36.811548 | UNKNOWN_LOCATION | SEVERE | Can not get assembly information for node card

CLUSTER 5
EVENTS: 1
  2005-08-04 10:58:22.010246 | R06-M1-ND | WARNING | Node card is not fully functional

CLUSTER 6
EVENTS: 2
  2005-08-09 10:40:46.749252 | R51-M1-ND | WARNING | Node card is not fully functional
  2005-08-09 10:53:17.485009 | R74-M0-N1 | WARNING | Node card is not fully functional

CLUSTER 7
EVENTS: 1
  2005-09-20 11:57:30.636832 | R05-M0-NA | INFO | Node card VPD check: U01 node in processor card slot J15 do not match. VPD ecid 04DE7DB80D7BFFFF04051B70D8D9, found 04DE7DF2D37BFFFF09081B6088D9

CLUSTER 8
EVENTS:

In [21]:
from incidentiq.context.patterns import repeated_message_patterns

In [22]:
patterns = repeated_message_patterns(context)

patterns

[{'type': 'repeated_message',
  'message': 'Node card is not fully functional',
  'occurrences': 6,
  'evidence_ids': [457, 1948, 1229, 1227, 1218, 620]},
 {'type': 'repeated_message',
  'message': 'Can not get assembly information for node card',
  'occurrences': 2,
  'evidence_ids': [522, 1204]}]

In [23]:
from incidentiq.context.patterns import(extract_patterns)

In [24]:
patterns = extract_patterns(context)

for pattern in patterns:
    print(pattern,"\n")

{'type': 'repeated_message', 'message': 'Node card is not fully functional', 'occurrences': 6, 'evidence_ids': [457, 1948, 1229, 1227, 1218, 620]} 

{'type': 'repeated_message', 'message': 'Can not get assembly information for node card', 'occurrences': 2, 'evidence_ids': [522, 1204]} 

{'type': 'affected_nodes', 'count': 10} 

{'type': 'severity_distribution', 'distribution': {'WARNING': 6, 'INFO': 2, 'SEVERE': 2}} 

{'type': 'temporal_cluster', 'event_count': 2, 'start': Timestamp('2005-08-09 10:40:46.749252'), 'end': Timestamp('2005-08-09 10:53:17.485009'), 'duration': Timedelta('0 days 00:12:30.735757')} 



In [25]:
from incidentiq.reasoning.models import (
    Observation,
    Hypothesis,
    IncidentAnalysis,
)


analysis = IncidentAnalysis(

    summary=(
        "The retrieved evidence shows a recurring node-card-related "
        "failure pattern across multiple nodes."
    ),

    observations=[

        Observation(
            statement=(
                "The message 'Node card is not fully functional' "
                "appears in 6 retrieved events."
            ),
            evidence_ids=[
                457,
                1948,
                1229,
                1227,
                1218,
                620,
            ],
        ),

        Observation(
            statement=(
                "The message 'Can not get assembly information for "
                "node card' appears in 2 retrieved events with SEVERE severity."
            ),
            evidence_ids=[
                522,
                1204,
            ],
        ),

        Observation(
            statement=(
                "The retrieved evidence involves 10 distinct nodes."
            ),
            evidence_ids=[
                457,
                1948,
                1229,
                1227,
                1218,
                620,
                1801,
                522,
                1413,
                1204,
            ],
        ),

        Observation(
            statement=(
                "Two node-card functionality failures occurred within "
                "approximately 13 minutes of each other on August 9, 2005."
            ),
            evidence_ids=[
                1227,
                1229,
            ],
        ),
    ],

    hypotheses=[

        Hypothesis(
            statement=(
                "The retrieved evidence suggests a recurring "
                "node-card-related failure pattern."
            ),
            evidence_ids=[
                457,
                1948,
                1229,
                1227,
                1218,
                620,
                522,
                1204,
            ],
            confidence=0.85,
        ),

        Hypothesis(
            statement=(
                "The repeated node-card functionality and assembly-information "
                "errors may indicate a broader node-card hardware or "
                "configuration issue."
            ),
            evidence_ids=[
                522,
                1204,
                457,
                1948,
                1229,
                1227,
            ],
            confidence=0.65,
        ),
    ],

    unknowns=[

        "The available evidence does not establish the underlying root cause.",

        "The retrieved events are spread across several months, so they "
        "cannot be assumed to represent one continuous incident.",

        "The evidence does not establish whether the affected nodes share "
        "a common hardware component or configuration.",

    ],
)


print(analysis)

summary='The retrieved evidence shows a recurring node-card-related failure pattern across multiple nodes.' observations=[Observation(statement="The message 'Node card is not fully functional' appears in 6 retrieved events.", evidence_ids=[457, 1948, 1229, 1227, 1218, 620]), Observation(statement="The message 'Can not get assembly information for node card' appears in 2 retrieved events with SEVERE severity.", evidence_ids=[522, 1204]), Observation(statement='The retrieved evidence involves 10 distinct nodes.', evidence_ids=[457, 1948, 1229, 1227, 1218, 620, 1801, 522, 1413, 1204]), Observation(statement='Two node-card functionality failures occurred within approximately 13 minutes of each other on August 9, 2005.', evidence_ids=[1227, 1229])] hypotheses=[Hypothesis(statement='The retrieved evidence suggests a recurring node-card-related failure pattern.', evidence_ids=[457, 1948, 1229, 1227, 1218, 620, 522, 1204], confidence=0.85), Hypothesis(statement='The repeated node-card function

In [26]:
patterns = extract_patterns(context)

for pattern in patterns:
    print(pattern)

{'type': 'repeated_message', 'message': 'Node card is not fully functional', 'occurrences': 6, 'evidence_ids': [457, 1948, 1229, 1227, 1218, 620]}
{'type': 'repeated_message', 'message': 'Can not get assembly information for node card', 'occurrences': 2, 'evidence_ids': [522, 1204]}
{'type': 'affected_nodes', 'count': 10}
{'type': 'severity_distribution', 'distribution': {'WARNING': 6, 'INFO': 2, 'SEVERE': 2}}
{'type': 'temporal_cluster', 'event_count': 2, 'start': Timestamp('2005-08-09 10:40:46.749252'), 'end': Timestamp('2005-08-09 10:53:17.485009'), 'duration': Timedelta('0 days 00:12:30.735757')}


In [27]:
from incidentiq.reasoning.analyzer import IncidentAnalyzer

analyzer = IncidentAnalyzer()

analysis = analyzer.analyze(
    query=query,
    context=context,
    patterns=patterns,
)

analysis

IncidentAnalysis(summary='Investigation results for: node card failure', observations=[Observation(statement="The message 'Node card is not fully functional' appears in 6 retrieved events.", evidence_ids=[457, 1948, 1229, 1227, 1218, 620]), Observation(statement="The message 'Can not get assembly information for node card' appears in 2 retrieved events.", evidence_ids=[522, 1204])], hypotheses=[Hypothesis(statement='The repeated node-card functionality and assembly-information errors may indicate a broader node-card hardware or configuration issue.', evidence_ids=[457, 1948, 1229, 1227, 1218, 620, 522, 1204], confidence=0.65)], unknowns=['The available evidence does not establish the underlying root cause.', 'The retrieved events span multiple months and cannot be assumed to represent one continuous incident.', 'The evidence does not establish whether the affected nodes share a common hardware component or configuration.'])

In [28]:
from incidentiq.reasoning.prompt import build_reasoning_prompt

prompt = build_reasoning_prompt(
    query,
    context,
    patterns,
)

print(prompt)

INCIDENT QUERY
node card failure

EVIDENCE
[457] 2005-06-28 09:53:39.479164 | R02-M1-NE | WARNING | Node card is not fully functional | 
[1948] 2005-12-06 10:05:04.300635 | R12-M0-NC | WARNING | Node card is not fully functional | 
[1229] 2005-08-09 10:53:17.485009 | R74-M0-N1 | WARNING | Node card is not fully functional | 
[1227] 2005-08-09 10:40:46.749252 | R51-M1-ND | WARNING | Node card is not fully functional | 
[1218] 2005-08-04 10:58:22.010246 | R06-M1-ND | WARNING | Node card is not fully functional | 
[620] 2005-07-08 23:15:03.798449 | R05-M0-N2 | WARNING | Node card is not fully functional | 
[1801] 2005-11-29 16:21:56.855025 | R67-M1-N7 | INFO | Node card VPD check: U01 node in processor card slot J05 do not match. VPD ecid 04D37DF2DE7BFFFF0D081AF0DAD2, found 04DD80740E2FFFFF0A0C19D0CEBD | 
[522] 2005-07-01 11:05:31.120732 | R37-M1-N4 | SEVERE | Can not get assembly information for node card | 
[1413] 2005-09-20 11:57:30.636832 | R05-M0-NA | INFO | Node card VPD check: U01 